<a href="https://colab.research.google.com/github/1900690/pear_ripeness_analysis/blob/main/pear_colerchart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ナシの撮影用カラーチャート作成ツール


##
<table style="border: none; border-collapse: collapse;">
  <tr style="border: none;">
    <td style="border: none; padding: 10px; vertical-align: top;">
      <img src="https://github.com/1900690/pear_ripeness_analysis/raw/main/images/manyual.png" width="400">
      <div style="text-align: center;">撮影方法の例</div>
    </td>
  </tr>
</table>

In [7]:
# @title カラーチャート付きPDF作成 { run: "auto" }

# @markdown ---
# @markdown **設定**
上半分の色 = "白" # @param ["黒", "白"]
色の割合 = 0.3 # @param {type:"slider", min:0.1, max:0.9, step:0.05}
PDFをダウンロードする = True # @param {type:"boolean"}
# @markdown ---

import subprocess
import sys
import os

# 日本語フォント対応
def install_and_import():
    try:
        import japanize_matplotlib
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "japanize-matplotlib", "-q"])
        import japanize_matplotlib

install_and_import()

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.backends.backend_pdf import PdfPages
import colorsys
import japanize_matplotlib
from google.colab import files

# データ定義
data = [
    {"色番号": "1", "H": 57, "S": 68, "V": 45},
    {"色番号": "2", "H": 55, "S": 56, "V": 42},
    {"色番号": "3", "H": 47, "S": 56, "V": 40},
    {"色番号": "4", "H": 42, "S": 66, "V": 43},
    {"色番号": "5", "H": 37, "S": 71, "V": 38},
    {"色番号": "6", "H": 28, "S": 65, "V": 41},
    {"色番号": "①", "H": 53, "S": 78, "V": 51},
    {"色番号": "②", "H": 50, "S": 88, "V": 60},
    {"色番号": "③", "H": 46, "S": 93, "V": 58},
    {"色番号": "④", "H": 47, "S": 94, "V": 62},
    {"色番号": "⑤", "H": 26, "S": 86, "V": 58},
    {"色番号": "⑥", "H": 33, "S": 94, "V": 69},
]
df = pd.DataFrame(data)

def hsv_to_rgb(h, s, v):
    return colorsys.hsv_to_rgb(h / 360.0, s / 100.0, v / 100.0)

df['RGB'] = df.apply(lambda row: hsv_to_rgb(row['H'], row['S'], row['V']), axis=1)

# 背景色と文字色の決定
bg_color = "black" if 上半分の色 == "黒" else "white"
text_color = "white" if 上半分の色 == "黒" else "black"

filename = 'nashi_color_chart_double_sided.pdf'

# 2ページのPDFを作成
with PdfPages(filename) as pdf:
    # ---------------------------
    # 1ページ目 (表面：カラーチャート)
    # ---------------------------
    fig, ax = plt.subplots(figsize=(11.69, 8.27), facecolor=bg_color)
    ax.set_facecolor(bg_color)

    n = len(df)
    width = 1.0 / n

    for i in range(n):
        x = i * width

        # 「色の割合」スライダーで高さを調整
        rect = patches.Rectangle(
            (x, 0.0), width, 色の割合,
            facecolor=df['RGB'].iloc[i],
            edgecolor=text_color,
            linewidth=1.5
        )
        ax.add_patch(rect)

        # 色番号を色の帯の少し上に配置（帯の高さに応じて自動調整）
        label_y = 色の割合 + (1.0 - 色の割合) * 0.1
        ax.text(x + width / 2, label_y, df['色番号'].iloc[i],
                fontsize=26, weight='bold', ha='center', va='center', color=text_color)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # 余白なし
    plt.subplots_adjust(left=0, right=1, bottom=0, top=1)

    # 1ページ目を保存
    pdf.savefig(fig, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)

    # ---------------------------
    # 2ページ目 (裏面：全面無地)
    # ---------------------------
    fig2, ax2 = plt.subplots(figsize=(11.69, 8.27), facecolor=bg_color)
    ax2.set_facecolor(bg_color)

    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('off')

    plt.subplots_adjust(left=0, right=1, bottom=0, top=1)

    # 2ページ目を保存
    pdf.savefig(fig2, dpi=300, facecolor=fig2.get_facecolor(), edgecolor='none')
    plt.close(fig2)


# --- 説明部分をテキスト出力 ---
print("="*70)
print(" 🍏【ナシ収穫適期カラーチャート (PDF両面印刷用) 】")
print("="*70)
print(f"※全2ページのPDFを生成しました（裏面・上半分の色：{上半分の色} / 色の割合：{色の割合*100:.0f}%）。")
print("\n[作り方]")
print("1. ダウンロードしたPDFを、A4用紙（横向き）に「両面印刷」かつ「余白なし（フチなし印刷）」で印刷します。")
print("   - 短辺とじ・長辺とじのどちらで設定しても問題ありません。")
print(f"   - これにより、表面にはカラーチャート、裏面は一面【{上半分の色}】になります。")
print("2. カラーチャートがある面が【内側】になるように、短辺同士を合わせて丸めます。")
print("   - 筒の外側（裏面）も指定した色になるため、遮光性が高まります。")
print("3. 端をテープ等で固定して綺麗な円柱を作れば完成です。")
print("="*70)


# チェックボックスがONの場合のみダウンロード処理を実行
if PDFをダウンロードする:
    print(f"\n[完了] PDFファイル '{filename}' をダウンロードしています...")
    try:
        files.download(filename)
    except Exception as e:
        print(f"ダウンロードに失敗しました。左側のフォルダアイコンから手動でダウンロードしてください。（エラー: {e}）")
else:
    print("\nPDFファイルが生成されました。ダウンロードは行われませんでした。")

 🍏【ナシ収穫適期カラーチャート (PDF両面印刷用) 】
※全2ページのPDFを生成しました（裏面・上半分の色：白 / 色の割合：30%）。

[作り方]
1. ダウンロードしたPDFを、A4用紙（横向き）に「両面印刷」かつ「余白なし（フチなし印刷）」で印刷します。
   - 短辺とじ・長辺とじのどちらで設定しても問題ありません。
   - これにより、表面にはカラーチャート、裏面は一面【白】になります。
2. カラーチャートがある面が【内側】になるように、短辺同士を合わせて丸めます。
   - 筒の外側（裏面）も指定した色になるため、遮光性が高まります。
3. 端をテープ等で固定して綺麗な円柱を作れば完成です。

[完了] PDFファイル 'nashi_color_chart_double_sided.pdf' をダウンロードしています...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>